# Traditional NLP — Complete Tutorial Notebook

**COMP 442/542 · Natural Language Processing**  
Tutorial by Tamta Kapanadze · Supervised by Prof. Gözde Gül Şahin

---

## Contents

| Section | Topic |
|---------|-------|
| **1** | Text Representation Basics — Bag of Words & N-grams |
| **2** | N-gram Language Models — Bigram probabilities |
| **2S** | Smoothing & Discounting — Add-1, Add-k, Backoff, Interpolation |
| **3** | Perplexity — Evaluating language models |

**Corpus:** *The Little Prince*, Antoine de Saint-Exupéry (1943) — public domain  
**Reference:** Jurafsky & Martin, *Speech and Language Processing*, Ch. 3

> **Note on efficiency:** Section 1 uses raw strings for readability.
> From Section 2 onwards we switch to **integer indices** — the standard practice in real NLP systems.

---
# Section 1: Text Representation Basics

Bag of Words · N-grams

> We use raw strings here for readability.

---
## 1.1 — Bag of Words (BoW)

The idea is simple: represent each sentence as a **vector of word counts**.  
We ignore word order — we just count how many times each word appears.

**Steps:**
1. Tokenize (split into words)
2. Build a shared vocabulary
3. Count occurrences of each word per sentence

In [1]:
# # Cell 3
# Our two sentences
s1 = "the food was not good the service was"
s2 = "the food was good the service was not"

# Step 1: Tokenize (split into words)
tokens_s1 = s1.split()
tokens_s2 = s2.split()

print("S1 tokens:", tokens_s1)
print("S2 tokens:", tokens_s2)

S1 tokens: ['the', 'food', 'was', 'not', 'good', 'the', 'service', 'was']
S2 tokens: ['the', 'food', 'was', 'good', 'the', 'service', 'was', 'not']


In [2]:
# # Cell 4
# Step 2: Build a shared vocabulary (sorted list of unique words)
vocab = sorted(set(tokens_s1 + tokens_s2))
print("Vocabulary:", vocab)

Vocabulary: ['food', 'good', 'not', 'service', 'the', 'was']


In [3]:
# # Cell 5
# Step 3: Count occurrences of each word in each sentence
def build_bow(tokens, vocab):
    return [tokens.count(word) for word in vocab]

bow_s1 = build_bow(tokens_s1, vocab)
bow_s2 = build_bow(tokens_s2, vocab)

print(f"{'Word':<12} {'S1':>4} {'S2':>4}")
print("-" * 22)
for word, c1, c2 in zip(vocab, bow_s1, bow_s2):
    print(f"{word:<12} {c1:>4} {c2:>4}")

Word           S1   S2
----------------------
food            1    1
good            1    1
not             1    1
service         1    1
the             2    2
was             2    2


In [4]:
# # Cell 6
# Are the two BoW vectors identical?
print("Are BoW vectors identical?", bow_s1 == bow_s2)

Are BoW vectors identical? True


### What does this mean?

S1 and S2 produce **identical BoW vectors** — even though one is negative and the other is (arguably) mixed.  
A classifier trained on BoW features would give **the same prediction** for both sentences.

> ⚠️ **Word order is completely lost in BoW.**

This is the core limitation we want to address.

---
## 1.2 — N-grams

**Idea:** Instead of individual words, look at sequences of N consecutive words.

| N | Name | Example from S1 |
|---|------|-----------------|
| 1 | Unigram | `"food"`, `"not"`, `"good"` |
| 2 | Bigram | `"was not"`, `"not good"` |
| 3 | Trigram | `"food was not"`, `"was not good"` |

N-grams capture **local word order** — they can tell `"was not good"` apart from `"was good"`.

In [5]:
# # Cell 9
# Extract n-grams from a list of tokens
def get_ngrams(tokens, n):
    #       join tokens[i : i+n] with a space
    return [" ".join(tokens[i:i+n]) for i in range(len(tokens) - n + 1)]

# Test on S1
print("--- S1 ---")
for n, name in [(1, "Unigrams"), (2, "Bigrams"), (3, "Trigrams")]:
    print(f"{name}: {get_ngrams(tokens_s1, n)}")

--- S1 ---
Unigrams: ['the', 'food', 'was', 'not', 'good', 'the', 'service', 'was']
Bigrams: ['the food', 'food was', 'was not', 'not good', 'good the', 'the service', 'service was']
Trigrams: ['the food was', 'food was not', 'was not good', 'not good the', 'good the service', 'the service was']


In [6]:
# # Cell 10
# Now compare bigrams of S1 and S2 side by side
bigrams_s1 = get_ngrams(tokens_s1, 2)
bigrams_s2 = get_ngrams(tokens_s2, 2)

print("Bigrams S1:", bigrams_s1)
print("Bigrams S2:", bigrams_s2)

# Which bigrams are unique to each sentence?
only_in_s1 = set(bigrams_s1) - set(bigrams_s2)
only_in_s2 = set(bigrams_s2) - set(bigrams_s1)

print("\nOnly in S1:", only_in_s1)
print("Only in S2:", only_in_s2)

Bigrams S1: ['the food', 'food was', 'was not', 'not good', 'good the', 'the service', 'service was']
Bigrams S2: ['the food', 'food was', 'was good', 'good the', 'the service', 'service was', 'was not']

Only in S1: {'not good'}
Only in S2: {'was good'}


In [18]:
#TODO print trigrams that appear only in S1 and only in S2

### Key observation

Even with bigrams, we can already see the difference:
- S1 contains `"not good"` → negative signal
- S2 contains `"was good"` → positive signal

Bigrams **partially** recover word order. But they still can't capture long-range dependencies like:

> *"The computer which I had just put into the machine room on the fifth floor **crashed**"*

The subject (*computer*) and its verb (*crashed*) are far apart — a bigram model would miss this.

---
## 1.3 — Mini Exercise

Try it yourself! Given the sentence below, extract its **bigrams** and **trigrams**.

> *"I love natural language processing"*

In [9]:
# # Cell 13
exercise = "I love natural language processing"

#TODO print out bigram and trigrams for the sentence
# ...

Bigrams: ['i love', 'love natural', 'natural language', 'language processing']
Trigrams: ['i love natural', 'love natural language', 'natural language processing']


---
## Section 1 Summary

| Method | Captures order? | Differentiates S1 vs S2? |
|--------|----------------|-------------------------|
| Bag of Words | ❌ | ❌ |
| Bigrams | Locally ✅ | ✅ |
| Trigrams | Better ✅ | ✅ |

➡️ **Next:** N-gram Language Models — we'll assign *probabilities* to word sequences and learn how to calculate P("I love NLP").

---
# Section 2: N-gram Language Models

Bigram probabilities · Chain rule · Zero probability problem

> ⚠️ We switch to integer indices partway through this section.

---
## 2.1 — Setup: Tokenize the Corpus

In [7]:
# # Cell 17
from collections import Counter

# Our corpus — The Little Prince (public domain)
corpus = (
    "the little prince went away he went to look at the roses "
    "he said to the roses you are not at all like my rose "
    "no one has tamed you and you have tamed no one "
    "my fox was like that once"
)

# Tokenize
tokens = corpus.split()

print(f"Total tokens : {len(tokens)}")
print(f"Unique words : {len(set(tokens))}")
print(f"Tokens       : {tokens}")

Total tokens : 42
Unique words : 28
Tokens       : ['the', 'little', 'prince', 'went', 'away', 'he', 'went', 'to', 'look', 'at', 'the', 'roses', 'he', 'said', 'to', 'the', 'roses', 'you', 'are', 'not', 'at', 'all', 'like', 'my', 'rose', 'no', 'one', 'has', 'tamed', 'you', 'and', 'you', 'have', 'tamed', 'no', 'one', 'my', 'fox', 'was', 'like', 'that', 'once']


---
## 2.2 — Count Unigrams

A **unigram count** tells us how many times each word appears in the corpus.  
We need this to calculate bigram probabilities later.

In [8]:
# # Cell 19
# Count unigrams
unigram_counts = Counter(tokens)

print("Most common words:")
for word, count in unigram_counts.most_common(10):
    print(f"  {word:<12} {count}")

Most common words:
  the          3
  you          3
  went         2
  he           2
  to           2
  at           2
  roses        2
  like         2
  my           2
  no           2


In [9]:
# # Cell 20
print("Count of 'he'     :", unigram_counts["he"])
print("Count of 'the'    :", unigram_counts["the"])
print("Count of 'little' :", unigram_counts["little"])

Count of 'he'     : 2
Count of 'the'    : 3
Count of 'little' : 1


---
## 2.3 — Count Bigrams

A **bigram** is a pair of consecutive words.  
We count how many times each pair appears in the corpus.

In [10]:
# # Cell 22
# # Extract all bigrams from the corpus
def get_bigrams(tokens):
    return list(zip(tokens, tokens[1:]))

bigrams = get_bigrams(tokens)
bigram_counts = Counter(bigrams)

print("Most common bigrams:")
for bigram, count in bigram_counts.most_common(8):
    print(f"  {str(bigram):<30} {count}")

Most common bigrams:
  ('the', 'roses')               2
  ('no', 'one')                  2
  ('the', 'little')              1
  ('little', 'prince')           1
  ('prince', 'went')             1
  ('went', 'away')               1
  ('away', 'he')                 1
  ('he', 'went')                 1


In [17]:
# # Cell 23
print("Count of ('he', 'said')    :", bigram_counts[("he", "said")])
print("Count of ('the', 'little') :", bigram_counts[("the", "little")])

Count of ('he', 'said')    : 1
Count of ('the', 'little') : 1


---
## ⚠️ Wait — This is Inefficient!

Everything we've done so far uses **raw strings** as dictionary keys.
For our tiny 42-token corpus this is fine — but imagine:

- Wikipedia: ~3 billion tokens, V ~= 1,000,000 words
- Storing string pairs `('the', 'little')` as bigram keys is **slow and memory-heavy**
- Comparing strings character by character is much slower than comparing integers

**In practice, every word is mapped to an integer index first.**  
All counts, probabilities, and lookups then work with integers instead of strings.

Let's switch to that representation now — and use it for everything going forward.

In [11]:
# # Cell 25
# Build word <-> index mappings
vocab    = sorted(set(tokens))           # sorted for reproducibility
word2idx = {w: i for i, w in enumerate(vocab)}
idx2word = {i: w for w, i in word2idx.items()}

V = len(vocab)
print(f"Vocabulary size: {V}")
print()
print("Sample mappings (word → index):")
for w in ["the", "little", "prince", "he", "rose"]:
    print(f"  '{w}' → {word2idx[w]}")

Vocabulary size: 28

Sample mappings (word → index):
  'the' → 23
  'little' → 10
  'prince' → 17
  'he' → 8
  'rose' → 18


In [12]:
# # Cell 26
# Convert token list from strings to integer indices
token_ids = [word2idx[w] for w in tokens]

print("Original tokens (strings):")
print(" ", tokens[:8])
print()
print("Token IDs (integers):")
print(" ", token_ids[:8])
print()
print("Much more compact — and integer comparisons are orders of magnitude faster than string comparisons.")

Original tokens (strings):
  ['the', 'little', 'prince', 'went', 'away', 'he', 'went', 'to']

Token IDs (integers):
  [23, 10, 17, 26, 4, 8, 26, 24]

Much more compact — and integer comparisons are orders of magnitude faster than string comparisons.


In [14]:
# # Cell 27
# Rebuild unigram and bigram counts using integer IDs
unigram_counts_idx = Counter(token_ids)
bigram_counts_idx  = Counter(zip(token_ids, token_ids[1:]))
total_tokens       = len(token_ids)

print("Unigram counts (index → count):")
for w in ["the", "he", "little"]:
    idx = word2idx[w]
    print(f"  '{w}' (id={idx}) → count={unigram_counts_idx[idx]}")

print()
print("Bigram counts (index pair → count):")
for w1, w2 in [("he", "said"), ("the", "little")]:
    i1, i2 = word2idx[w1], word2idx[w2]
    print(f"  ('{w1}','{ w2}') = ({i1},{i2}) → count={bigram_counts_idx[(i1,i2)]}")

Unigram counts (index → count):
  'the' (id=23) → count=3
  'he' (id=8) → count=2
  'little' (id=10) → count=1

Bigram counts (index pair → count):
  ('he','said') = (8,20) → count=1
  ('the','little') = (23,10) → count=1


---
## 2.4 — From Counts to Probabilities

The bigram probability formula is:

$$P(w_i \mid w_{i-1}) = \frac{\text{Count}(w_{i-1},\ w_i)}{\text{Count}(w_{i-1})}$$

We divide the bigram count by the unigram count of the first word.

In [15]:
# # Cell 29
# Bigram probability using integer indices
def bigram_prob(i1, i2):
    if unigram_counts_idx[i1] == 0:
        return 0.0
    return bigram_counts_idx[(i1, i2)] / unigram_counts_idx[i1]

# Example: P(went | he)
i_he   = word2idx['he']
i_went = word2idx['went']
p = bigram_prob(i_he, i_went)
print(f"P(went | he)  = P(id={i_went} | id={i_he})")
print(f"             = {bigram_counts_idx[(i_he, i_went)]} / {unigram_counts_idx[i_he]} = {p:.4f}")

P(went | he)  = P(id=26 | id=8)
             = 1 / 2 = 0.5000


In [16]:
# # Cell 30
i_said   = word2idx['said']
i_little = word2idx['little']
i_the    = word2idx['the']

p1 = bigram_prob(i_he, i_said)
p2 = bigram_prob(i_the, i_little)

print(f"P(said   | he)  = {p1:.4f}")
print(f"P(little | the) = {p2:.4f}")

P(said   | he)  = 0.5000
P(little | the) = 0.3333


---
## 2.5 — Calculate Sentence Probability

Using the **chain rule** + **bigram assumption**, the probability of a sentence is:

$$P(w_1 w_2 \ldots w_n) \approx P(w_1) \times \prod_{i=2}^{n} P(w_i \mid w_{i-1})$$

Let's calculate **P("he went to look")** step by step.

In [17]:
# # Cell 32
def sentence_prob(sentence):
    words = sentence.lower().split()
    ids   = [word2idx[w] for w in words]
    # Start with unigram probability of first word
    prob  = unigram_counts_idx[ids[0]] / total_tokens
    print(f"  P({words[0]}) [id={ids[0]}] = {unigram_counts_idx[ids[0]]} / {total_tokens} = {prob:.4f}")
    for j in range(1, len(ids)):
        i1, i2 = ids[j-1], ids[j]
        p = bigram_prob(i1, i2)
        print(f"  P({words[j]} | {words[j-1]}) [ids=({i1},{i2})] = {bigram_counts_idx[(i1,i2)]} / {unigram_counts_idx[i1]} = {p:.4f}")
        prob *= p
    return prob

print("Calculating P('he went to look'):")
prob = sentence_prob('he went to look')
print(f"\nFinal probability = {prob:.6f}")

Calculating P('he went to look'):
  P(he) [id=8] = 2 / 42 = 0.0476
  P(went | he) [ids=(8,26)] = 1 / 2 = 0.5000
  P(to | went) [ids=(26,24)] = 1 / 2 = 0.5000
  P(look | to) [ids=(24,11)] = 1 / 2 = 0.5000

Final probability = 0.005952


---
## 2.6 — The Zero Probability Problem

What happens when a bigram has never been seen in our corpus?

In [18]:
print("Calculating P('the little prince said'):")
prob = sentence_prob('the little prince said')
print(f"\nFinal probability = {prob:.6f}")

print()
print("⚠️  'said' never followed 'prince' in our corpus.")
print("    P(said | prince) = 0  →  the entire sentence probability = 0")
print("    This is the Zero Probability Problem.")
print("    Solution: Smoothing (next section!)")

Calculating P('the little prince said'):
  P(the) [id=23] = 3 / 42 = 0.0714
  P(little | the) [ids=(23,10)] = 1 / 3 = 0.3333
  P(prince | little) [ids=(10,17)] = 1 / 1 = 1.0000
  P(said | prince) [ids=(17,20)] = 0 / 1 = 0.0000

Final probability = 0.000000

⚠️  'said' never followed 'prince' in our corpus.
    P(said | prince) = 0  →  the entire sentence probability = 0
    This is the Zero Probability Problem.
    Solution: Smoothing (next section!)


---
## 2.7 — Mini Exercise

Now it's your turn!  
Calculate **P("he said to the")** using the functions we built.

Expected steps:
- P(he)
- P(said | he)
- P(to | said)
- P(the | to)

In [ ]:
# # Cell 36
# TODO
print("Calculating P('he said to the'):")
# prob = ...
# print(f"\nFinal probability = {prob:.6f}")

Calculating P('he said to the'):
  P(he) [id=8] = 2 / 42 = 0.0476
  P(said | he) [ids=(8,20)] = 1 / 2 = 0.5000
  P(to | said) [ids=(20,24)] = 1 / 1 = 1.0000
  P(the | to) [ids=(24,23)] = 1 / 2 = 0.5000

Final probability = 0.011905


---
## Section 2 Summary

| Concept | What we did |
|---------|-------------|
| Unigram counts | Counted how often each word appears |
| Bigram counts | Counted consecutive word pairs |
| Bigram probability | P(wᵢ \| wᵢ₋₁) = Count(wᵢ₋₁, wᵢ) / Count(wᵢ₋₁) |
| Sentence probability | Multiply unigram × bigram probabilities |
| Zero probability | Unseen bigrams → P = 0 → need smoothing |

➡️ **Next:** Perplexity — how do we measure how good a language model is?

---
# Section 2S: Smoothing & Discounting

Add-1 · Add-k · Stupid Backoff · Interpolation

> All functions use integer indices internally.

---
## Setup: Corpus & Counts

In [19]:
from collections import Counter

# Corpus — The Little Prince (public domain)
corpus = (
    "the little prince went away he went to look at the roses "
    "he said to the roses you are not at all like my rose "
    "no one has tamed you and you have tamed no one "
    "my fox was like that once"
)

tokens = corpus.split()

# Word <-> index mappings (more efficient than raw strings)
vocab    = sorted(set(tokens))
word2idx = {w: i for i, w in enumerate(vocab)}
idx2word = {i: w for w, i in word2idx.items()}

# All counts use integer indices
token_ids          = [word2idx[w] for w in tokens]
unigram_counts     = Counter(token_ids)
bigram_counts      = Counter(zip(token_ids, token_ids[1:]))
total_tokens       = len(token_ids)
V                  = len(vocab)

# Helper: look up by word string (converts to index internally)
def w2i(w): return word2idx[w]

print(f"Total tokens : {total_tokens}")
print(f"Vocabulary   : {V}")
print(f"Unique bigrams: {len(bigram_counts)}")

Total tokens : 42
Vocabulary   : 28
Unique bigrams: 39


---
## 2S.1 — Baseline: MLE (No Smoothing)

$$P_{MLE}(w_i \mid w_{i-1}) = \frac{\text{Count}(w_{i-1},\ w_i)}{\text{Count}(w_{i-1})}$$

In [20]:
def mle_prob(w1, w2):
    i1, i2 = w2i(w1), w2i(w2)
    if unigram_counts[i1] == 0: return 0.0
    return bigram_counts[(i1, i2)] / unigram_counts[i1]

def add1_prob(w1, w2):
    i1, i2 = w2i(w1), w2i(w2)
    return (bigram_counts[(i1, i2)] + 1) / (unigram_counts[i1] + V)

def addk_prob(w1, w2, k):
    i1, i2 = w2i(w1), w2i(w2)
    return (bigram_counts[(i1, i2)] + k) / (unigram_counts[i1] + k * V)

def stupid_backoff(w1, w2, alpha=0.4):
    i1, i2 = w2i(w1), w2i(w2)
    if bigram_counts[(i1, i2)] > 0:
        return bigram_counts[(i1, i2)] / unigram_counts[i1]
    return alpha * (unigram_counts[i2] / total_tokens)

def interpolated_prob(w1, w2, lambda1=0.7, lambda2=0.3):
    i1, i2 = w2i(w1), w2i(w2)
    bigram_p  = mle_prob(w1, w2)
    unigram_p = unigram_counts[i2] / total_tokens
    return lambda1 * bigram_p + lambda2 * unigram_p

print('✅ All probability functions use integer indices internally')

✅ All probability functions use integer indices internally


In [21]:
# Sentence probability with MLE
def sentence_prob_mle(sentence):
    words = sentence.lower().split()
    i0    = w2i(words[0])
    if i0 == -1: return 0.0
    prob  = unigram_counts[i0] / total_tokens
    for i in range(1, len(words)):
        prob *= mle_prob(words[i-1], words[i])
    return prob

print(f"P('he went to look')         = {sentence_prob_mle('he went to look'):.6f}")
print(f"P('the little prince said')  = {sentence_prob_mle('the little prince said'):.6f}  ← zero!")

P('he went to look')         = 0.005952
P('the little prince said')  = 0.000000  ← zero!


---
## 2S.2 — Add-1 (Laplace) Smoothing

Add 1 to every bigram count — so nothing is ever zero.

$$P_{Add-1}(w_i \mid w_{i-1}) = \frac{\text{Count}(w_{i-1},\ w_i) + 1}{\text{Count}(w_{i-1}) + V}$$

In [22]:
# # Cell 45
def add1_prob(w1, w2):
    i1, i2 = w2i(w1), w2i(w2)
    if i1 == -1 or i2 == -1: return 1 / (total_tokens + V)
    return (bigram_counts[(i1, i2)] + 1) / (unigram_counts[i1] + V)

# Compare MLE vs Add-1
test_bigrams = [
    ("prince", "went"),
    ("he",     "said"),
    ("the",    "roses"),
    ("prince", "said"),  # unseen
    ("rose",   "was"),   # unseen
]

print(f"{'Bigram':<20} {'MLE':>8} {'Add-1':>10}")
print("-" * 42)
for w1, w2 in test_bigrams:
    mle  = mle_prob(w1, w2)
    add1 = add1_prob(w1, w2)
    flag = " ← was zero!" if mle == 0 else ""
    print(f"({w1}, {w2}){'':>{18-len(w1)-len(w2)-4}} {mle:>8.4f} {add1:>10.4f}{flag}")

Bigram                    MLE      Add-1
------------------------------------------
(prince, went)       1.0000     0.0690
(he, said)           0.5000     0.0667
(the, roses)         0.6667     0.0968
(prince, said)       0.0000     0.0345 ← was zero!
(rose, was)          0.0000     0.0345 ← was zero!


In [23]:
# # Cell 46
def sentence_prob_mle(sentence):
    words = sentence.lower().split()
    i0    = w2i(words[0])
    if i0 == -1: return 0.0
    prob  = unigram_counts[i0] / total_tokens
    for i in range(1, len(words)):
        prob *= mle_prob(words[i-1], words[i])
    return prob

def sentence_prob_add1(sentence):
    words = sentence.lower().split()
    i0    = w2i(words[0])
    if i0 == -1: return 0.0
    prob  = unigram_counts[i0] / total_tokens
    for i in range(1, len(words)):
        prob *= add1_prob(words[i-1], words[i])
    return prob

print(f"P('he went to look')        MLE   = {sentence_prob_mle('he went to look'):.6f}")
print(f"P('he went to look')        Add-1 = {sentence_prob_add1('he went to look'):.6f}")
print()
print(f"P('the little prince said') MLE   = {sentence_prob_mle('the little prince said'):.6f}  ← zero")
print(f"P('the little prince said') Add-1 = {sentence_prob_add1('the little prince said'):.6f}  ← no longer zero!")

P('he went to look')        MLE   = 0.005952
P('he went to look')        Add-1 = 0.000014

P('the little prince said') MLE   = 0.000000  ← zero
P('the little prince said') Add-1 = 0.000011  ← no longer zero!


---
## 2S.3 — Add-k Smoothing

Instead of adding 1, add a smaller fraction **k** — less aggressive than Add-1.

$$P_{Add-k}(w_i \mid w_{i-1}) = \frac{\text{Count}(w_{i-1},\ w_i) + k}{\text{Count}(w_{i-1}) + kV}$$

Common values: k = 0.5, k = 0.1, k = 0.01

In [37]:
# # Cell 48
def addk_prob(w1, w2, k):
    i1, i2 = w2i(w1), w2i(w2)
    if i1 == -1 or i2 == -1: return k / (total_tokens + k * V)
    return (bigram_counts[(i1, i2)] + k) / (unigram_counts[i1] + k * V)

# Compare different values of k
w1, w2 = "prince", "said"  # unseen bigram
print(f"Unseen bigram: ({w1}, {w2})")
print(f"  MLE    : {mle_prob(w1, w2):.6f}")
for k in [1.0, 0.5, 0.1, 0.01]:
    print(f"  Add-{k:<4}: {addk_prob(w1, w2, k):.6f}")

print()
w1, w2 = "he", "said"  # seen bigram
print(f"Seen bigram: ({w1}, {w2})")
print(f"  MLE    : {mle_prob(w1, w2):.6f}")
for k in [1.0, 0.5, 0.1, 0.01]:
    print(f"  Add-{k:<4}: {addk_prob(w1, w2, k):.6f}")

Unseen bigram: (prince, said)
  MLE    : 0.000000
  Add-1.0 : 0.034483
  Add-0.5 : 0.033333
  Add-0.1 : 0.026316
  Add-0.01: 0.007812

Seen bigram: (he, said)
  MLE    : 0.500000
  Add-1.0 : 0.066667
  Add-0.5 : 0.093750
  Add-0.1 : 0.229167
  Add-0.01: 0.442982


Notice: as k gets smaller, Add-k approaches MLE for seen bigrams, but still gives non-zero probability to unseen ones.

---
## 2S.4 — Stupid Backoff

Simple and effective for large corpora (Brants et al., 2007).  
No discounting — just fall back to a lower-order model if the bigram is unseen.

$$S(w_i \mid w_{i-1}) = \begin{cases} \frac{\text{Count}(w_{i-1},\ w_i)}{\text{Count}(w_{i-1})} & \text{if Count}(w_{i-1}, w_i) > 0 \\ 0.4 \cdot \frac{\text{Count}(w_i)}{N} & \text{otherwise} \end{cases}$$

> Note: This is a **score**, not a true probability (does not sum to 1).

In [24]:
# # Cell 51
def stupid_backoff(w1, w2, alpha=0.4):
    i1, i2 = w2i(w1), w2i(w2)
    if i1 == -1 or i2 == -1: return 0.0
    if bigram_counts[(i1, i2)] > 0:
        return bigram_counts[(i1, i2)] / unigram_counts[i1]
    else:
        return alpha * (unigram_counts[i2] / total_tokens)

print(f"{'Bigram':<22} {'MLE':>8} {'Backoff':>10}")
print("-" * 44)
for w1, w2 in test_bigrams:
    mle = mle_prob(w1, w2)
    sb  = stupid_backoff(w1, w2)
    flag = " ← backed off" if mle == 0 else ""
    print(f"({w1}, {w2}){'':>{20-len(w1)-len(w2)-4}} {mle:>8.4f} {sb:>10.4f}{flag}")

Bigram                      MLE    Backoff
--------------------------------------------
(prince, went)         1.0000     1.0000
(he, said)             0.5000     0.5000
(the, roses)           0.6667     0.6667
(prince, said)         0.0000     0.0095 ← backed off
(rose, was)            0.0000     0.0095 ← backed off


---
## 2S.5 — Simple Interpolation

Combine trigram, bigram, and unigram models with weights that sum to 1.

$$\hat{P}(w_i \mid w_{i-2}w_{i-1}) = \lambda_1 P(w_i \mid w_{i-2}w_{i-1}) + \lambda_2 P(w_i \mid w_{i-1}) + \lambda_3 P(w_i)$$

where $\lambda_1 + \lambda_2 + \lambda_3 = 1$

Since our corpus is small we'll use **bigram + unigram interpolation** (λ₁ + λ₂ = 1).

In [25]:
# # Cell 53
def interpolated_prob(w1, w2, lambda1=0.7, lambda2=0.3):
    i1, i2 = w2i(w1), w2i(w2)
    if i1 == -1 or i2 == -1: return 0.0
    assert abs(lambda1 + lambda2 - 1.0) < 1e-9, "Lambdas must sum to 1"
    bigram_p  = mle_prob(w1, w2)
    unigram_p = unigram_counts[i2] / total_tokens
    return lambda1 * bigram_p + lambda2 * unigram_p

# Compare all methods
print(f"{'Bigram':<22} {'MLE':>8} {'Add-1':>8} {'Backoff':>9} {'Interp':>9}")
print("-" * 60)
for w1, w2 in test_bigrams:
    mle    = mle_prob(w1, w2)
    add1   = add1_prob(w1, w2)
    sb     = stupid_backoff(w1, w2)
    interp = interpolated_prob(w1, w2)
    print(f"({w1}, {w2}){'':>{20-len(w1)-len(w2)-4}} {mle:>8.4f} {add1:>8.4f} {sb:>9.4f} {interp:>9.4f}")

Bigram                      MLE    Add-1   Backoff    Interp
------------------------------------------------------------
(prince, went)         1.0000   0.0690    1.0000    0.7143
(he, said)             0.5000   0.0667    0.5000    0.3571
(the, roses)           0.6667   0.0968    0.6667    0.4810
(prince, said)         0.0000   0.0345    0.0095    0.0071
(rose, was)            0.0000   0.0345    0.0095    0.0071


---
## 2S.6 — Full Comparison: Sentence Probabilities

Let's compare all methods on two sentences — one seen, one with an unseen bigram.

In [26]:
# # Cell 55
def sentence_prob_generic(sentence, prob_fn):
    """Calculate sentence probability using any bigram probability function."""
    words = sentence.lower().split()
    i0    = w2i(words[0])
    if i0 == -1: return 0.0
    prob  = unigram_counts[i0] / total_tokens
    for i in range(1, len(words)):
        prob *= prob_fn(words[i-1], words[i])
    return prob

sentences = [
    "he went to look",
    "the little prince said",
]

methods = [
    ("MLE",           mle_prob),
    ("Add-1",         add1_prob),
    ("Add-0.1",       lambda w1, w2: addk_prob(w1, w2, 0.1)),
    ("Backoff",       stupid_backoff),
    ("Interpolation", interpolated_prob),
]

for sentence in sentences:
    print(f"\nP('{sentence}')")
    print("-" * 45)
    for name, fn in methods:
        p = sentence_prob_generic(sentence, fn)
        flag = "  ← ZERO" if p == 0 else ""
        print(f"  {name:<16}: {p:.8f}{flag}")


P('he went to look')
---------------------------------------------
  MLE             : 0.00595238
  Add-1           : 0.00001411
  Add-0.1         : 0.00057311
  Backoff         : 0.00595238
  Interpolation   : 0.00225687

P('the little prince said')
---------------------------------------------
  MLE             : 0.00000000  ← ZERO
  Add-1           : 0.00001096
  Add-0.1         : 0.00010320
  Backoff         : 0.00022676
  Interpolation   : 0.00008676


---
## 2S.7 — Mini Exercise

Try calculating the smoothed probability of a new sentence: **"the fox was like that"**

This sentence has:
- `(fox, was)` — seen ✅
- `(was, like)` — seen ✅  
- `(like, that)` — seen ✅

So MLE should work fine here. Let's verify, then compare with smoothed methods.

In [27]:
# # Cell 57
sentence = "the fox was like that"

print(f"P('{sentence}')")
print("-" * 45)
for name, fn in methods:
    p = sentence_prob_generic(sentence, fn)
    print(f"  {name:<16}: {p:.8f}")

P('the fox was like that')
---------------------------------------------
  MLE             : 0.00000000
  Add-1           : 0.00000073
  Add-0.1         : 0.00002365
  Backoff         : 0.00034014
  Interpolation   : 0.00009204


---
## Summary

| Method | Zero prob? | Formula | Notes |
|--------|-----------|---------|-------|
| MLE | ❌ Yes | Count(w₁,w₂) / Count(w₁) | Baseline |
| Add-1 | ✅ No | (Count+1) / (Count+V) | Too aggressive |
| Add-k | ✅ No | (Count+k) / (Count+kV) | Tune k |
| Stupid Backoff | ✅ No | MLE or 0.4 × unigram | Not a true prob |
| Interpolation | ✅ No | λ₁×bigram + λ₂×unigram | Tune λ weights |

➡️ **Next:** Perplexity — how do we measure how good a language model actually is?

---
# Section 3: Perplexity

Evaluating language models · Log probabilities · Comparing methods

> All functions use integer indices internally.

---
## Setup: Reusing our corpus and models from Sections 2 & 2S

In [28]:
# # Cell 61
from collections import Counter

# Corpus — The Little Prince (public domain)
corpus = (
    "the little prince went away he went to look at the roses "
    "he said to the roses you are not at all like my rose "
    "no one has tamed you and you have tamed no one "
    "my fox was like that once"
)

tokens = corpus.split()

# Word <-> index mappings
vocab    = sorted(set(tokens))
word2idx = {w: i for i, w in enumerate(vocab)}
idx2word = {i: w for w, i in word2idx.items()}
def w2i(w): return word2idx.get(w, -1)  # -1 for OOV words

# All counts use integer indices
token_ids      = [word2idx[w] for w in tokens]
unigram_counts = Counter(token_ids)
bigram_counts  = Counter(zip(token_ids, token_ids[1:]))
total_tokens   = len(token_ids)
V              = len(vocab)

print(f'✅ Setup complete')
print(f'   Corpus: {total_tokens} tokens, V={V}')
print(f'   All functions use integer indices internally')

✅ Setup complete
   Corpus: 42 tokens, V=28
   All functions use integer indices internally


In [29]:
# # Cell 62
def mle_prob(w1, w2):
    i1, i2 = w2i(w1), w2i(w2)
    if i1 == -1 or i2 == -1: return 0.0
    if unigram_counts[i1] == 0: return 0.0
    return bigram_counts[(i1, i2)] / unigram_counts[i1]

def add1_prob(w1, w2):
    i1, i2 = w2i(w1), w2i(w2)
    if i1 == -1 or i2 == -1: return 1 / (total_tokens + V)  # tiny prob for OOV
    return (bigram_counts[(i1, i2)] + 1) / (unigram_counts[i1] + V)

def addk_prob(w1, w2, k=0.1):
    i1, i2 = w2i(w1), w2i(w2)
    if i1 == -1 or i2 == -1: return k / (total_tokens + k * V)
    return (bigram_counts[(i1, i2)] + k) / (unigram_counts[i1] + k * V)

def stupid_backoff(w1, w2, alpha=0.4):
    i1, i2 = w2i(w1), w2i(w2)
    if i1 == -1 or i2 == -1: return 0.0
    if bigram_counts[(i1, i2)] > 0:
        return bigram_counts[(i1, i2)] / unigram_counts[i1]
    return alpha * (unigram_counts[i2] / total_tokens)

def interpolated_prob(w1, w2, lambda1=0.7, lambda2=0.3):
    i1, i2 = w2i(w1), w2i(w2)
    if i1 == -1 or i2 == -1: return 0.0
    return lambda1 * mle_prob(w1, w2) + lambda2 * (unigram_counts[i2] / total_tokens)

print('✅ Setup complete')
print(f'   Corpus: {total_tokens} tokens, V={V}')
print(f'   All functions use integer indices internally')

✅ Setup complete
   Corpus: 42 tokens, V=28
   All functions use integer indices internally


---
## 3.1 — Sentence Log Probability

In practice we use **log probabilities** to avoid numerical underflow  
(multiplying many small numbers together quickly becomes 0.0 in floating point).

$$\log P(w_1 \ldots w_n) = \log P(w_1) + \sum_{i=2}^{n} \log P(w_i \mid w_{i-1})$$

In [30]:
import math
# # Cell 64
def sentence_log_prob(sentence, prob_fn):
    """Returns log probability of a sentence under a bigram model."""
    words = sentence.lower().split()
    # First word: unigram probability
    i0 = w2i(words[0])
    p0 = unigram_counts[i0] / total_tokens if i0 != -1 else 0.0
    if p0 == 0:
        return float('-inf')
    log_p = math.log(p0)
    # Remaining words: bigram probabilities
    for i in range(1, len(words)):
        p = prob_fn(words[i-1], words[i])
        if p == 0:
            return float('-inf')
        log_p += math.log(p)
    return log_p

# Test on a seen sentence
s = 'he went to look at the roses'
lp_mle  = sentence_log_prob(s, mle_prob)
lp_add1 = sentence_log_prob(s, add1_prob)

print(f"Sentence: '{s}'")
print(f"  MLE  log P = {lp_mle:.4f}   P = {math.exp(lp_mle):.8f}")
print(f"  Add1 log P = {lp_add1:.4f}   P = {math.exp(lp_add1):.8f}")

Sentence: 'he went to look at the roses'
  MLE  log P = -6.2226   P = 0.00198413
  Add1 log P = -18.8862   P = 0.00000001


In [31]:
# Test on a sentence with an unseen bigram
s2 = "the little prince said"
lp_mle2  = sentence_log_prob(s2, mle_prob)
lp_add12 = sentence_log_prob(s2, add1_prob)

print(f"Sentence: '{s2}'")
print(f"  MLE  log P = {lp_mle2}  ← -inf because of zero bigram!")
print(f"  Add1 log P = {lp_add12:.4f}   P = {math.exp(lp_add12):.10f}")

Sentence: 'the little prince said'
  MLE  log P = -inf  ← -inf because of zero bigram!
  Add1 log P = -11.4213   P = 0.0000109591


---
## 3.2 — Perplexity Formula

For a test set W of N words:

$$PP(W) = P(w_1 w_2 \ldots w_N)^{-1/N}$$

In log space:

$$\log PP(W) = -\frac{1}{N} \sum_{i=1}^{N} \log P(w_i \mid w_{i-1})$$

$$PP(W) = e^{-\frac{1}{N} \sum \log P(w_i \mid w_{i-1})}$$

In [32]:
# # Cell 67
def perplexity(test_sentences, prob_fn):
    """
    Compute perplexity of a model on a list of test sentences.
    Returns inf if any sentence has zero probability under the model.
    """
    #   1. Sum up log probabilities of all sentences
    #   2. Count total number of words
    #   3. Return exp(-total_log / total_words)
    total_log   = 0.0
    total_words = 0
    for s in test_sentences:
        lp = sentence_log_prob(s, prob_fn)
        if lp == float('-inf'):
            return float('inf')
        total_log   += lp
        total_words += len(s.split())
    return math.exp(-total_log / total_words)

# Test on a single sentence
s = "he went to look at the roses"
print(f"Sentence: '{s}'")
print(f"  Words: {len(s.split())}")
print(f"  MLE  PPL = {perplexity([s], mle_prob):.4f}")
print(f"  Add1 PPL = {perplexity([s], add1_prob):.4f}")

Sentence: 'he went to look at the roses'
  Words: 7
  MLE  PPL = 2.4325
  Add1 PPL = 14.8505


---
## 3.3 — Comparing Models on Test Sentences

Let's evaluate all our smoothing methods on the same test set.

In [33]:
# # Cell 69
# Test sentences — from our Little Prince corpus
test_sentences = [
    "he went to look at the roses",
    "the little prince went away",
    "he said to the roses you are not",
]

methods = [
    ("MLE",           mle_prob),
    ("Add-1",         add1_prob),
    ("Add-0.1",       lambda w1, w2: addk_prob(w1, w2, k=0.1)),
    ("Backoff",       stupid_backoff),
    ("Interpolation", interpolated_prob),
]

print("Per sentence perplexity:")
print()
col_w = 36
header = f"{'Sentence':<{col_w}}" + "".join(f"{name:>14}" for name, _ in methods)
print(header)
print("-" * (col_w + 14 * len(methods)))
for s in test_sentences:
    row = f"'{s}'"[:col_w-1].ljust(col_w)
    for name, fn in methods:
        ppl = perplexity([s], fn)
        cell = "INF" if ppl == float('inf') else f"{ppl:.2f}"
        row += f"{cell:>14}"
    print(row)

print()
print("Overall perplexity (all sentences combined):")
print()
print(f"{'Method':<16} {'PPL':>10}")
print("-" * 30)
for name, fn in methods:
    ppl = perplexity(test_sentences, fn)
    flag = "  ← INFINITE (zero prob)" if ppl == float('inf') else ""
    print(f"{name:<16} {ppl:>10.4f}{flag}")

Per sentence perplexity:

Sentence                                       MLE         Add-1       Add-0.1       Backoff Interpolation
----------------------------------------------------------------------------------------------------------
'he went to look at the roses'                2.43         14.85          4.95          2.43          3.20
'the little prince went away'                 2.43         14.69          5.21          2.43          3.18
'he said to the roses you are not'            2.29         14.87          4.84          2.29          3.04

Overall perplexity (all sentences combined):

Method                  PPL
------------------------------
MLE                  2.3728
Add-1               14.8171
Add-0.1              4.9704
Backoff              2.3728
Interpolation        3.1306


### Why does MLE win on these sentences?

MLE gives the lowest perplexity here because **all three test sentences are from the training corpus**.
MLE assigns them very high probability — but would fail completely on any unseen sentence.

This is why we always evaluate on a **held-out test set** that was **not** used for training!

---
## 3.4 — Unseen Test Sentences

Now let's use sentences that were **not** in our training corpus.

In [34]:
# # Cell 72
unseen = [
    "the prince looked at the rose",   # 'looked' is OOV
    "he looked at the fox",            # 'looked' is OOV
    "the little fox went away",        # all words known, new combination
]

print(f"{'Sentence':<38} {'MLE':>8}  {'Add-1':>8}  {'Add-k':>8}  {'Backoff':>10}  {'Interp':>10}")
print("-" * 82)
for s in unseen:
    results = []
    for fn in [mle_prob, add1_prob, addk_prob, stupid_backoff, interpolated_prob]:
        results.append(perplexity([s], fn))
    row = [f"{r:.2f}" if r != float('inf') else "INF" for r in results]
    print(f"'{s}'  {row[0]:>8}  {row[1]:>8}  {row[2]:>8}  {row[3]:>10}  {row[4]:>10}")

print()
print("MLE → INF on any sentence with an unseen bigram (zero probability).")
print("Smoothed models assign small non-zero probability to unseen pairs,")
print("keeping perplexity finite. 'looked' is fully OOV — not in training vocab.")


Sentence                                    MLE     Add-1     Add-k     Backoff      Interp
----------------------------------------------------------------------------------
'the prince looked at the rose'       INF     31.56     58.78         INF         INF
'he looked at the fox'       INF     34.35     63.92         INF         INF
'the little fox went away'       INF     19.39     13.60       13.59       17.41

MLE → INF on any sentence with an unseen bigram (zero probability).
Smoothed models assign small non-zero probability to unseen pairs,
keeping perplexity finite. 'looked' is fully OOV — not in training vocab.


**Results breakdown:**

- **MLE** fails on all sentences — any unseen bigram or OOV word gives P=0
- **Backoff** and **Interpolation** survive only when all words are in the vocabulary
  (e.g. *"the little fox went away"* — all words known, just new combinations)
  but fail on OOV words like *"looked"* which never appeared in our corpus
- **Add-1** and **Add-k** handle both unseen bigrams AND OOV words
  by assigning a small non-zero probability to every possible word pair

**Key distinction:**
```
Unseen bigram (known words) → Backoff/Interpolation can help
Unseen word (OOV)           → only Add-1 / Add-k survive
```

---
## 3.5 — Mini Exercise

Calculate the perplexity of each model on the sentence below.  
This sentence has one unseen bigram — can you find which one?

> *"the rose tamed the fox"*

In [35]:
exercise = "the rose tamed the fox"

# First, find the unseen bigram
words = exercise.split()
print("Bigrams in the sentence:")
for j in range(len(words)-1):
    w1, w2 = words[j], words[j+1]
    i1, i2 = w2i(w1), w2i(w2)
    count = bigram_counts[(i1, i2)] if i1 != -1 and i2 != -1 else 0
    flag  = "  ← UNSEEN" if count == 0 else f"  count={count}"
    print(f"  ({w1}, {w2}){flag}")

print()
print(f"{'Method':<16} {'PPL':>12}")
print("-" * 32)
for name, fn in methods:
    ppl = perplexity([exercise], fn)
    flag = "  ← INFINITE" if ppl == float('inf') else ""
    print(f"{name:<16} {ppl:>12.4f}{flag}")

Bigrams in the sentence:
  (the, rose)  ← UNSEEN
  (rose, tamed)  ← UNSEEN
  (tamed, the)  ← UNSEEN
  (the, fox)  ← UNSEEN

Method                    PPL
--------------------------------
MLE                       inf  ← INFINITE
Add-1                 25.9223
Add-0.1               38.6191
Backoff               49.0396
Interpolation         61.7302


---
## Section 3 Summary

| Concept | Key point |
|---------|----------|
| Perplexity | Inverse probability of test set, normalized by N |
| Lower is better | Model is less surprised = better fit |
| Log probabilities | Avoid numerical underflow in practice |
| MLE on training data | PPL looks great — but fails on unseen data |
| Smoothing + PPL | Higher PPL on seen data, but handles unseen gracefully |
| Same vocabulary | Only valid to compare models with same V |

**WSJ reference results (Jurafsky & Martin, Ch. 3):**

| Model | PPL |
|-------|-----|
| Unigram | 962 |
| Bigram | 170 |
| Trigram | 109 |

More context → lower perplexity → better model.

---
# Section 4: KenLM — Industrial-Strength Language Models

Real-world NLP · ARPA format · KenLM Python API · Comparing with our implementation

> Our implementation in Sections 2–3 was built from scratch for learning purposes.
> In practice, the field uses optimised toolkits that handle billion-token corpora
> and state-of-the-art smoothing. **KenLM** (Heafield, 2011) is the most widely used.

---
## Why use a dedicated LM toolkit?

| What our implementation does | What KenLM adds |
|------------------------------|-----------------|
| Add-1 / Add-k smoothing | Modified Kneser-Ney (state of the art) |
| Python dicts for counts | Memory-mapped binary indexes |
| Works on tiny corpora | Handles billions of tokens efficiently |
| Manually written ARPA | `lmplz` trains directly from raw text |


---
## The ARPA format

ARPA is the standard plain-text format for n-gram language models — KenLM reads and writes it,
and it's what `lmplz` outputs. Understanding it helps you see exactly what probabilities the
model is using.

```
\data\
ngram 1=4       # number of unique unigrams
ngram 2=3       # number of unique bigrams

\1-grams:
-0.301  the     0.0    # log10 P(the)  [TAB]  word  [TAB]  backoff weight
-0.477  cat     0.0
-0.602  sat     0.0
-0.602  </s>    0.0

\2-grams:
-0.301  the cat         # log10 P(cat | the)
-0.301  cat sat
-0.301  sat </s>

\end\
```

Key points:
- Probabilities are **log base 10** — `-0.301` = log₁₀(0.5)
- Every line: `log10_prob [TAB] word(s) [TAB] backoff_weight`
- Backoff weights allow the model to fall back to lower-order n-grams for unseen sequences


In [36]:
# # Cell 79 (takes a few minutes!)
%%bash
# Install kenlm Python bindings
pip install kenlm -q

# Build lmplz and build_binary from source (Colab / Linux)
if ! command -v lmplz &> /dev/null; then
    echo "Building KenLM binaries from source..."
    apt-get install -y cmake libboost-all-dev libeigen3-dev zlib1g-dev -qq 2>/dev/null

    git clone --depth 1 https://github.com/kpu/kenlm.git /tmp/kenlm_src 2>/dev/null
    mkdir -p /tmp/kenlm_src/build
    cd /tmp/kenlm_src/build
    cmake .. -DCMAKE_BUILD_TYPE=Release -DKENLM_MAX_ORDER=6
    make -j$(nproc) lmplz build_binary
    cp /tmp/kenlm_src/build/bin/lmplz /usr/local/bin/
    cp /tmp/kenlm_src/build/bin/build_binary /usr/local/bin/
    echo "Done — lmplz and build_binary installed."
else
    echo "lmplz already available: $(which lmplz)"
fi

lmplz --version 2>&1 | head -1


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 427.5/427.5 kB 7.6 MB/s eta 0:00:00
Building KenLM binaries from source...
Selecting previously unselected package javascript-common.
(Reading database ... 118194 files and directories currently installed.)
Preparing to unpack .../000-javascript-common_11+nmu1_all.deb ...
Unpacking javascript-common (11+nmu1) ...
Selecting previously unselected package libboost1.74-tools-dev.
Preparing to unpack .../001-libboost1.74-tools-dev_1.74.0-14ubuntu3_amd64.deb ...
Unpacking libboost1.74-tools-dev (1.74.0-14ubuntu3) ...
Selecting previously unselected package libboost-tools-dev.
Preparing to unpack .../002-libboost-tools-dev_1.74.0.3ubuntu7_amd64.deb ...
Unpacking libboost-tools-dev (1.74.0.3ubuntu7) ...
Selecting previously unselected package libboost-atomic1.74.0:amd64.
Preparing to unpack .../003-libboost-atomic1.74.0_1.74.0-14ubuntu3_amd64.deb ...
Unpacking libboost-atomic1.74.0:amd64 (1.74.0-14ubuntu3) ...
Selecting previously unselected package

CMake Warning (dev) at CMakeLists.txt:101 (find_package):
  Policy CMP0167 is not set: The FindBoost module is removed.  Run "cmake
  --help-policy CMP0167" for policy details.  Use the cmake_policy command to
  set the policy and suppress this warning.

This warning is for project developers.  Use -Wno-dev to suppress it.



---
## 4.1 — Training with `lmplz`

`lmplz` reads raw text (one sentence per line) and outputs an ARPA file with
**Modified Kneser-Ney smoothing**. The cell below writes the corpus to disk,
runs `lmplz`, and falls back to a Python-generated Add-1 ARPA only if `lmplz`
isn't found.

> **`--discount_fallback`** is needed for tiny corpora like ours — without it,
> KN's discount estimation can fail when counts are very low.


In [37]:
# # Cell 81
import math, os, shutil, subprocess
from collections import Counter

corpus = (
    "the little prince went away he went to look at the roses "
    "he said to the roses you are not at all like my rose "
    "no one has tamed you and you have tamed no one "
    "my fox was like that once"
)
tokens = corpus.split()
vocab_words = sorted(set(tokens))
V = len(vocab_words)

sent_tokens = ["<s>"] + tokens + ["</s>"]
unigram = Counter(sent_tokens)
bigram  = Counter(zip(sent_tokens, sent_tokens[1:]))
N = len(sent_tokens)

# Write corpus file (needed by lmplz)
train_path = "/tmp/lp_train.txt"
with open(train_path, "w") as f:
    f.write("the little prince went away he went to look at the roses\n")
    f.write("he said to the roses you are not at all like my rose\n")
    f.write("no one has tamed you and you have tamed no one\n")
    f.write("my fox was like that once\n")

arpa_path = "/tmp/lp_kenlm.arpa"

if shutil.which("lmplz"):
    print("lmplz found — training with Kneser-Ney smoothing...")
    result = subprocess.run(
        ["lmplz", "-o", "2", "--discount_fallback"],
        stdin=open(train_path),
        stdout=open(arpa_path, "w"),
        stderr=subprocess.PIPE,
        text=True
    )
    print(result.stderr.strip())
    print(f"\nARPA written by lmplz to {arpa_path} (Kneser-Ney smoothed)")
    USED_KNESER_NEY = True
else:
    print("lmplz not found — generating ARPA from Python with Add-1 smoothing.")
    print("(Install via: conda install -c conda-forge kenlm for true Kneser-Ney)")

    def unigram_log10(w):
        return math.log10((unigram[w] + 1) / (N + V))

    def bigram_log10(w1, w2):
        return math.log10((bigram[(w1, w2)] + 1) / (unigram[w1] + V))

    unk_log10 = math.log10(1 / (N + V))
    all_words = sorted(set(sent_tokens))

    lines = [
        "\\data\\",
        f"ngram 1={len(all_words) + 1}",
        f"ngram 2={len(bigram)}",
        "",
        "\\1-grams:",
        f"{unk_log10:.4f}\t<unk>\t0.0000",
    ]
    for w in all_words:
        lines.append(f"{unigram_log10(w):.4f}\t{w}\t0.0000")
    lines += ["", "\\2-grams:"]
    for (w1, w2) in sorted(bigram.keys()):
        lines.append(f"{bigram_log10(w1,w2):.4f}\t{w1} {w2}")
    lines += ["", "\\end\\"]

    with open(arpa_path, "w") as f:
        f.write("\n".join(lines))

    print(f"ARPA written to {arpa_path} (Add-1 smoothed)")
    USED_KNESER_NEY = False

print(f"\nSmoothing used: {'Kneser-Ney (lmplz)' if USED_KNESER_NEY else 'Add-1 (Python fallback)'}")


lmplz found — training with Kneser-Ney smoothing...
=== 1/5 Counting and sorting n-grams ===
Reading /tmp/lp_train.txt
----5---10---15---20---25---30---35---40---45---50---55---60---65---70---75---80---85---90---95--100
****************************************************************************************************
Unigram tokens 42 types 31
=== 2/5 Calculating and sorting adjusted counts ===
Chain sizes: 1:372 2:10884667801
Substituting fallback discounts for order 1: D1=0.5 D2=1 D3+=1.5
Statistics:
1 31 D1=0.529412 D2=1.60294 D3+=1.94118
2 44 D1=0.5 D2=1 D3+=1.5
Memory estimate for binary LM:
type       B
probing 1608 assuming -p 1.5
probing 1736 assuming -r models -p 1.5
trie     995 without quantization
trie    1898 assuming -q 8 -b 8 quantization 
trie     995 assuming -a 22 array pointer compression
trie    1898 assuming -a 22 -q 8 -b 8 array pointer compression and quantization
=== 3/5 Calculating and sorting initial probabilities ===
Chain sizes: 1:372 2:704
=== 4/5 Calcula

---
## 4.2 — Binary format

`build_binary` converts the ARPA file to a compact binary format that loads ~10×
faster and uses less RAM. It was installed alongside `lmplz` in the build step above.
For our 42-token corpus the difference is negligible, but on a real corpus (millions
of n-grams) you'd always convert to binary before loading.


In [38]:
# # Cell 83
import shutil, subprocess

arpa_path = "/tmp/lp_kenlm.arpa"
bin_path  = "/tmp/lp_kenlm.bin"

if shutil.which("build_binary"):
    subprocess.run(["build_binary", arpa_path, bin_path], check=True)
    model_path = bin_path
    print(f"Binary model written to {bin_path}")
else:
    model_path = arpa_path
    print("build_binary not found — loading ARPA directly.")

print(f"Will load: {model_path}")


Binary model written to /tmp/lp_kenlm.bin
Will load: /tmp/lp_kenlm.bin


---
## 4.3 — Loading the model and querying sentence probability

`model.score(s)` returns log₁₀ P(sentence); `model.perplexity(s)` returns PPL
normalised by token count. Both include sentence-boundary markers internally.


In [39]:
# # Cell 85
import kenlm

model = kenlm.Model(model_path)
print(f"KenLM model loaded from {model_path}")
print()

test_sentences = [
    "he went to look at the roses",
    "the little prince went away",
    "he said to the roses you are not",
]

print(f"{'Sentence':<42} {'log10 P':>9}  {'PPL':>8}")
print("-" * 63)
for s in test_sentences:
    lp  = model.score(s)
    ppl = model.perplexity(s)
    print(f"'{s}'  {lp:>9.4f}  {ppl:>8.2f}")


KenLM model loaded from /tmp/lp_kenlm.bin

Sentence                                     log10 P       PPL
---------------------------------------------------------------
'he went to look at the roses'    -4.4369      3.59
'the little prince went away'    -4.1870      4.99
'he said to the roses you are not'    -5.7950      4.40


---
## 4.4 — Word-level scores with `full_scores()`

`full_scores()` returns a per-token breakdown — useful for spotting which words
the model finds most surprising.


In [40]:
# # Cell 87
sentence = "he went to look at the roses"
words    = sentence.split()

print(f"Token-level breakdown for: '{sentence}'")
print()
print(f"  {'Word':<12} {'log10 P':>9}  {'P':>10}  {'n-gram order':>13}  {'OOV?':>5}")
print("  " + "-" * 55)

for i, (log10p, ngram_len, oov) in enumerate(model.full_scores(sentence)):
    word = words[i] if i < len(words) else "</s>"
    prob = 10 ** log10p
    print(f"  {word:<12} {log10p:>9.4f}  {prob:>10.6f}  {ngram_len:>13}  {'yes' if oov else 'no':>5}")

print()
print("n-gram order = 1 → model fell back to unigram")
print("n-gram order = 2 → full bigram was used")


Token-level breakdown for: 'he went to look at the roses'

  Word           log10 P           P   n-gram order   OOV?
  -------------------------------------------------------
  he             -0.8533    0.140185              2     no
  went           -0.5765    0.265185              2     no
  to             -0.5765    0.265185              2     no
  look           -0.5751    0.266020              2     no
  at             -0.2880    0.515185              2     no
  the            -0.5643    0.272705              2     no
  roses          -0.4567    0.349354              2     no
  </s>           -0.5466    0.284069              2     no

n-gram order = 1 → model fell back to unigram
n-gram order = 2 → full bigram was used


---
## 4.5 — Comparing KenLM (Kneser-Ney) with our Add-1

KenLM trained with `lmplz` uses **Modified Kneser-Ney** smoothing — the industry standard.
The key difference from Add-1: instead of adding a flat count to every bigram, KN discounts
observed counts and redistributes mass based on how *versatile* a word is across contexts.
Words that appear in many different contexts get higher unigram backoff weight.

On sentences from the training corpus, KN should give noticeably lower PPL than Add-1.


In [41]:
# # Cell 89
import math
from collections import Counter

# Rebuild our Add-1 model from Section 2S
corpus = (
    "the little prince went away he went to look at the roses "
    "he said to the roses you are not at all like my rose "
    "no one has tamed you and you have tamed no one "
    "my fox was like that once"
)
tokens    = corpus.split()
vocab     = sorted(set(tokens))
V         = len(vocab)
word2idx  = {w: i for i, w in enumerate(vocab)}
def w2i(w): return word2idx.get(w, -1)

token_ids      = [word2idx[w] for w in tokens]
unigram_counts = Counter(token_ids)
bigram_counts  = Counter(zip(token_ids, token_ids[1:]))
total_tokens   = len(token_ids)

def add1_prob(w1, w2):
    i1, i2 = w2i(w1), w2i(w2)
    if i1 == -1 or i2 == -1: return 1 / (total_tokens + V)
    return (bigram_counts[(i1, i2)] + 1) / (unigram_counts[i1] + V)

def our_ppl(sentence):
    words = sentence.lower().split()
    i0 = w2i(words[0])
    p0 = unigram_counts[i0] / total_tokens if i0 != -1 else 0.0
    if p0 == 0: return float('inf')
    lp = math.log(p0)
    for i in range(1, len(words)):
        p = add1_prob(words[i-1], words[i])
        if p == 0: return float('inf')
        lp += math.log(p)
    N = len(words)
    return math.exp(-lp / N)

print(f"{'Sentence':<40} {'Ours (Add-1)':>13}  {'KenLM':>10}")
print("-" * 68)
for s in test_sentences:
    ours   = our_ppl(s)
    theirs = model.perplexity(s)
    print(f"'{s}'  {ours:>13.2f}  {theirs:>10.2f}")

print()
if USED_KNESER_NEY:
    print("KenLM uses Kneser-Ney — much lower PPL than Add-1 on training sentences.")
    print("KN discounts seen bigrams and redistributes mass more intelligently than Add-1.")
else:
    print("Both use Add-1 (Python fallback). Small gap from KenLM sentence markers.")


Sentence                                  Ours (Add-1)       KenLM
--------------------------------------------------------------------
'he went to look at the roses'          14.85        3.59
'the little prince went away'          14.69        4.99
'he said to the roses you are not'          14.87        4.40

KenLM uses Kneser-Ney — much lower PPL than Add-1 on training sentences.
KN discounts seen bigrams and redistributes mass more intelligently than Add-1.


---
## 4.6 — Mini exercise: unseen sentences

Now test both on sentences **not** in the training corpus.
With real Kneser-Ney, KenLM should handle unseen combinations better than Add-1 —
KN's backoff to unigrams is smarter than Add-1's flat +1 to every count.

'looked' is fully OOV (not in vocabulary at all) — watch how each model handles it.


In [72]:
# # Cell 91
unseen = [
    "the prince looked at the rose",   # 'looked' is OOV
    "he looked at the fox",            # 'looked' is OOV
    "the little fox went away",        # all words known, new combination
]

print(f"{'Sentence':<38} {'Ours (Add-1)':>13}  {'KenLM (KN)':>12}")
print("-" * 68)
for s in unseen:
    ours   = our_ppl(s)
    theirs = model.perplexity(s)
    ours_s = f"{ours:.2f}" if ours != float('inf') else "INF"
    print(f"'{s}'  {ours_s:>13}  {theirs:>12.2f}")

print()
if USED_KNESER_NEY:
    print("KN handles unseen combinations better — it backs off to smarter unigram")
    print("estimates rather than Add-1's uniform +1 pseudocount.")
    print("For OOV words ('looked'), both use a small <unk> probability.")
else:
    print("Both use Add-1 here (Python fallback). Install via conda for true KN.")


Sentence                                Ours (Add-1)    KenLM (KN)
--------------------------------------------------------------------
'the prince looked at the rose'          31.56        220.25
'he looked at the fox'          34.35        209.63
'the little fox went away'          19.39        260.54

KN handles unseen combinations better — it backs off to smarter unigram
estimates rather than Add-1's uniform +1 pseudocount.
For OOV words ('looked'), both use a small <unk> probability.


Let's try a bigger corpus and 5-grams.

In [58]:
from collections import Counter

corpus = corpus = """It is a truth universally acknowledged that a single man in possession of a good fortune must be in want of a wife.
However little known the feelings or views of such a man may be on his first entering a neighbourhood this truth is so well fixed in the minds of the surrounding families that he is considered as the rightful property of some one or other of their daughters.
My dear Mr Bennet said his lady to him one day have you heard that Netherfield Park is let at last.
Mr Bennet replied that he had not.
But it is returned she for Mrs Long has just been here and she told me all about it.
Mr Bennet made no answer.
Do not you want to know who has taken it cried his wife impatiently.
You want to tell me and I have no objection to hearing it.
This was invitation enough.
Why my dear you must know Mrs Long says that Netherfield is taken by a young man of large fortune from the north of England.
What is his name.
Bingley.
Is he married or single.
Oh single my dear to be sure a single man of large fortune four or five thousand a year what a fine thing for our girls.
How so can it affect them.
My dear Mr Bennet how can you be so tiresome you must know that I am thinking of his marrying one of them.
Is that his design in settling here.
Design nonsense how can you talk so but it is very likely that he may fall in love with one of them and therefore you must visit him as soon as he comes.
I see no occasion for that you and the girls may go or you may send them by themselves which perhaps will be still better than as you are as handsome as any of them Mr Bingley might like you the best of the party.
My dear you flatter me I certainly have had my share of beauty but I do not pretend to be anything extraordinary now when a woman has five grown up daughters she ought to give over thinking of her own beauty.
In such cases a woman has not often much beauty to think of.
But my dear you must indeed go and see Mr Bingley when he comes into the neighbourhood.
It is more than I engage for I assure you.
But consider your daughters only think what an establishment it would be for one of them.
Sir William and Lady Lucas are determined to go merely on that account for in general you know they visit no new comers indeed you must go for it will be impossible for us to visit him if you do not.
You are over scrupulous surely I dare say Mr Bingley will be very glad to see you and I will send a few lines by you to assure him of my hearty consent to his marrying whichever he chooses of our girls though I must throw in a good word for my little Lizzy.
I desire you will do no such thing Lizzy is not a bit better than the others and I am sure she is not half so handsome as Jane nor half so good humoured as Lydia but you are always giving her the preference.
They have none of them much to recommend them they are all silly and ignorant like other girls but Lizzy has something more of quickness than her sisters.
Mr Bennet how can you abuse your own children so you take delight in vexing me you have no compassion on my poor nerves.
You mistake me my dear I have a high respect for your nerves they are my old friends I have heard you mention them with consideration these twenty years at least.
Ah you do not know what I suffer.
But I hope you will get over it and live to see many young men of four thousand a year come into the neighbourhood.
It will be no use to us if twenty such should come since you will not visit them.
Depend upon it my dear that when there are twenty I will visit them all."""

In [59]:
# ── Step 1: write corpus as-is
if isinstance(corpus, str):
    lines = corpus.strip().split("\n")
else:
    lines = corpus

with open("/tmp/corpus_unk.txt", "w") as f:
    f.write("\n".join(lines))

print(f"Lines: {len(lines)}")
!wc -w /tmp/corpus_unk.txt

Lines: 34
713 /tmp/corpus_unk.txt


In [60]:
# ── Step 2: train with --vocab_estimate and --prune ───────────────────────
# lmplz handles <unk> internally — rare words not seen at test time
# automatically get the <unk> probability. No manual replacement needed.

result = subprocess.run([
    "lmplz", "-o", "5",
    "--text", "/tmp/corpus_unk.txt",
    "--arpa", "/tmp/model_5gram.arpa",
    "--prune", "0", "0", "1",
    "--discount_fallback",   # safe for any corpus size
], capture_output=True, text=True)

print(result.stderr)
print("Return code:", result.returncode)

=== 1/5 Counting and sorting n-grams ===
Reading /tmp/corpus_unk.txt
----5---10---15---20---25---30---35---40---45---50---55---60---65---70---75---80---85---90---95--100
****************************************************************************************************
Unigram tokens 713 types 305
=== 2/5 Calculating and sorting adjusted counts ===
Chain sizes: 1:3660 2:1061918400 3:1991097216 4:3185755392 5:4645893632
Substituting fallback discounts for order 3: D1=0.5 D2=1 D3+=1.5
Substituting fallback discounts for order 4: D1=0.5 D2=1 D3+=1.5
Statistics:
1 305 D1=0.692833 D2=1.1686 D3+=1.92226
2 650 D1=0.892537 D2=1.10746 D3+=2.10746
3 20/689 D1=0.968481 D2=1.47174 D3+=3
4 8/670 D1=0.5 D2=1 D3+=1.5
5 2/643 D1=0.5 D2=1 D3+=1.5
Memory estimate for binary LM:
type    kB
probing 23 assuming -p 1.5
probing 28 assuming -r models -p 1.5
trie    13 without quantization
trie    16 assuming -q 8 -b 8 quantization 
trie    13 assuming -a 22 array pointer compression
trie    16 assuming -a 22

In [61]:
# ── Step 3: build binary ──────────────────────────────────────────────────
subprocess.run([
    "build_binary",
    "/tmp/model_5gram.arpa",
    "/tmp/model_5gram.bin"
], check=True)
print("Binary ready.")

Binary ready.


In [71]:
# ── Step 4: compute perplexity ────────────────────────────────────────────
import kenlm, math

model = kenlm.Model("/tmp/model_5gram.bin")

# Test sentences — replace with your own
test_sentences = [
    # Seen — from the corpus directly
    "It is a truth universally acknowledged that a single man in possession of a good fortune must be in want of a wife.",
    "Mr Bennet made no answer.",
    "Is he married or single.",

    # Paraphrased — same ideas, different wording
    "A wealthy single man is always expected to seek a wife.",
    "He did not respond to her question.",

    # Unseen — plausible but never in the text
    "The young ladies were delighted to hear of the new arrival.",
    "She had never considered the possibility of such an arrangement.",

    # Odd / low probability — should have high PPL
    "Fortune good a man single acknowledged truth universally.",
]

# Per-sentence PPL
print(f"{'Sentence':<45} {'PPL':>8}")
print("-" * 55)
for s in test_sentences:
    print(f"'{s}'  {model.perplexity(s):>8.2f}")

# Corpus-level PPL (all sentences together)
total_log, total_words = 0.0, 0
for s in test_sentences:
    words = s.split()
    total_log   += model.score(s) * math.log(10)  # convert log10 → ln
    total_words += len(words)

corpus_ppl = math.exp(-total_log / total_words)
print(f"\nCorpus-level PPL: {corpus_ppl:.2f}")

Sentence                                           PPL
-------------------------------------------------------
'It is a truth universally acknowledged that a single man in possession of a good fortune must be in want of a wife.'     21.95
'Mr Bennet made no answer.'     17.11
'Is he married or single.'     21.33
'A wealthy single man is always expected to seek a wife.'    159.75
'He did not respond to her question.'    263.50
'The young ladies were delighted to hear of the new arrival.'    234.60
'She had never considered the possibility of such an arrangement.'    255.89
'Fortune good a man single acknowledged truth universally.'    249.46

Corpus-level PPL: 132.90


## 4.7 — Your Turn
Now try bulding a trigram model from the same corpus and evaluating the same sentences for PPL

In [73]:
# TODO

---
## Section 4 Summary

| Concept | Key point |
|---------|-----------|
| ARPA format | Standard plain-text LM — log₁₀ probs, tab-separated |
| `lmplz -o 2 --discount_fallback` | Train bigram model with Kneser-Ney from raw text |
| `build_binary model.arpa model.bin` | Convert to binary — ~10× faster load, less RAM |
| `kenlm.Model(path)` | Load ARPA or binary; accepts both formats |
| `model.score(s)` | Returns log₁₀ P(sentence), including `<s>`/`</s>` markers |
| `model.perplexity(s)` | Returns PPL normalised by token count |
| `model.full_scores(s)` | Per-token log₁₀ prob + n-gram order used + OOV flag |
| Kneser-Ney vs Add-1 | KN gives much lower PPL — discounts counts and uses versatility-based backoff |
| `--discount_fallback` | Required for small corpora where KN discount estimation can fail |

**Useful flags for larger corpora:**
- `lmplz -o 3` — trigram model
- `lmplz -o 5` — 5-gram (common for speech recognition)
- Always run `build_binary` before loading in production
